```python
# Copyright (c) 2026 Louis Gevers
# SPDX-License-Identifier: BSD-3-Clause
```

# Legged Robots Project - Task 3: DRL on Stairs

**Goal:** train a policy that makes the Go2 walk forward up and down a stairs reliably.

A flat-terrain policy fails on stairs as it has neve seen a step, and cannot see one coming.
This notebook changes the terrain to stairs, and introduces new components to train a more robust policy:

| Component | Why | Section |
| --- | --- | --- |
| Terrain scan | the policy perceives the steps ahead (exteroception) | 3.1b |
| Observation noise | real sensors are noisy, a policy trained on perfect data may not transfer | 3.1b |
| Domain randomization | the real robot never exactly matches the simulation model | 3.1f |
| Curriculum learning | robots start on easy stairs and only get harder ones once they succeed | 3.1g |


Start from your Task 1 configuration and adapt it: most of your observations, actions, rewards and terminations carry over!
There is no analysis notebook for this task: the goal is a policy that crosses stairs reliably, checked qualitatively in the viewer (3.4).

**Time budget.** Stairs are harder than flat ground and training takes longer!
Check your environment in the viewer (3.1i) before training, and plan your GPU time, especially close to the deadline.

In [ ]:
%load_ext autoreload
%autoreload 2

import math
import copy
import lr_project # Before mjlab import

from mjlab.envs import ManagerBasedRlEnvCfg
from mjlab.rl import RslRlModelCfg, RslRlPpoAlgorithmCfg, RslRlOnPolicyRunnerCfg

from mjlab.managers.action_manager import ActionTermCfg
from mjlab.managers.command_manager import CommandTermCfg
from mjlab.managers.curriculum_manager import CurriculumTermCfg
from mjlab.managers.event_manager import EventTermCfg
from mjlab.managers.observation_manager import ObservationTermCfg, ObservationGroupCfg
from mjlab.managers.reward_manager import RewardTermCfg
from mjlab.managers.termination_manager import TerminationTermCfg
from mjlab.managers.scene_entity_config import SceneEntityCfg
from mjlab.utils.noise import UniformNoiseCfg

from lr_project import mdp, scene, play, train, evaluate, registry

## 3.1 Building the environment configuration

The structure is the same as in Task 1.
The scene now contains a generated stairs terrain and a terrain scan sensor (see `src/lr_project/scene.py`), and the environment gets one new part: the curriculum (3.1g).

### 3.1a Commands

This task is **forward locomotion only**: `rel_forward_envs=1.0` gives every robot a forward command with $v_y = \omega_z = 0$.
Leave `lin_vel_y` and `ang_vel_z` as they are: they are overridden anyway.

Things to think about:
- Which speeds are realistic on stairs?
- Should robots sometimes be commanded to stand still on a staircase?

In [ ]:
def make_commands(play: bool) -> dict[str, CommandTermCfg]:
    return {
        "joystick": mdp.UniformVelocityCommandCfg(
            resampling_time_range=(1e3, 1e3), # TODO
            rel_standing_envs=0.0, # TODO
            rel_forward_envs=1.0, # IMPORTANT! This notebook does FORWARD locomotion only!
            ranges=mdp.UniformVelocityCommandCfg.Ranges(
                lin_vel_x=(-1.0, 1.0), # TODO
                lin_vel_y=(-1.0, 1.0), # Fix for mjlab viewer (min and max have to be different)...
                ang_vel_z=(-1.0, 1.0), # ...rel_forward_envs will override this. Do not change.
            ),
            entity_name="robot",
            debug_vis=play,
        )
    }


### 3.1b Observations

Two changes compared to Task 1.

**Terrain scan.**
The sensor `"terrain_scan"` casts a grid of rays (1.6 m $\times$ 1.0 m around the base, every 0.2 m) down onto the terrain.
`mdp.height_scan` returns, for each ray, the height of the base above the terrain at that point.
This enables the policy to see steps ahead.
You can see the rays in the viewer.

**Observation noise.**
Each term can now get uniform noise with `noise=UniformNoiseCfg(n_min=..., n_max=...)`, added to the observation at every step.
Noise is only applied in groups with `enable_corruption=True`.
Choose noise levels that roughly match the real sensors: joint encoders are precise, IMU angular velocities less so, and an elevation map can be very noisy (several centimetres).

Things to think about:
- Can the policy learn without the terrain scan (i.e., blind locomotion)? How would that policy differ?
- Should all observations, including the command, be noisy for robustness?
- Should the critic receive noisy observations?
- How would a terrain scan be obtained on a real robot?

In [ ]:
def make_observations(play: bool) -> dict[str, ObservationGroupCfg]:
    actor = {
        # Example
        "command": ObservationTermCfg(
            func=mdp.generated_command,
            params={"command_name": "joystick"},
            noise=UniformNoiseCfg(
                n_min=0.0,
                n_max=0.0,
            ),  # TODO: should the command have noise? Set noise=None to remove noise
        ),
        # TODO: add the terms the policy needs, with appropriate sensor noise
    }

    critic = {
        **actor,  # the critic sees everything the actor sees...
        # TODO: ...plus optional privileged terms (only available in simulation)
    }

    return {
        "actor": ObservationGroupCfg(
            terms=actor,
            enable_corruption=True,  # TODO: should the actor have noise on?
        ),
        "critic": ObservationGroupCfg(
            terms=critic,
            enable_corruption=True,  # TODO: should the critic have noise on?
        ),
    }

### 3.1c Actions

Same action space as in Task 1.

Things to think about:
- Does the action scale that worked on flat ground allow the legs to lift high enough for a step?

In [ ]:
def make_actions(play: bool) -> dict[str, ActionTermCfg]:
    return {
        "joint_targets": mdp.JointPositionActionCfg(
            entity_name="robot",
            actuator_names=(".*"), # all 12 joints
            use_default_offset=True, # TODO: should targets be offsets from the default standing pose?
            scale=0.0, # TODO: radians of joint motion per unit of action
        ),
    }

### 3.1d Rewards

Start from your Task 1 rewards, then tune each term against the new task.
Some penalties that made sense on flat ground might work against going on stairs.
For instance, does it make sense to penalize vertical base velocity when climbing up a stair?

Things to think about:
- Which penalties should be reduced or removed on stairs?
- Watch `Episode_Reward/<term>` and the terrain level (3.3), which dominates?

In [ ]:
def make_rewards(play: bool) -> dict[str, RewardTermCfg]:
    return {
        # Example
        "track_linear_velocity": RewardTermCfg(
            func=mdp.track_linear_velocity,
            weight=1.0,  # TODO: tune
            params={"command_name": "joystick", "std": math.sqrt(0.25)},
        ),
        # TODO: add penalties that make the motion smooth, stable and efficient.
    }

### 3.1e Terminations

Same principle as in Task 1: a time-out, plus at least one failure termination that detects a fall.
We just add one more termination, when the robot reaches the end of their terrain (`terrain_edge_reached`).

In [ ]:
def make_terminations(play: bool) -> dict[str, TerminationTermCfg]:
    return {
        # Example
        "time_out": TerminationTermCfg(
            func=mdp.time_out,
            time_out=True, # TODO: is this a failure or not?
        ),
        "terrain_edge_reached": TerminationTermCfg(
            func=mdp.terrain_edge_reached,
            time_out=True, # TODO: is this a failure or not?
        ),
        # TODO: add at least one failure termination that detects a fall
    }


### 3.1f Events

**Initial state.**
Each robot starts at the centre of a staircase (on its top or bottom platform, 1.5 m wide).
Choose the initial pose range so robots do not spawn partly inside a step.

**Domain randomization.**
The simulation model is never exactly the real robot.
Modern deep reinforcement learning methods rely on domain randomization, where physical properties of the robot and environment are randomized.
The policy has to learn to work for the whole range of robots with different robots, and, hopefully, for the real one.

Here you randomize the **centre of mass of the base**, with `mdp.body_com_offset`:

- `operation`: how the sampled value is applied: `"add"` to the nominal value, `"scale"` it, or replace it (`"abs"`)
- `ranges`: (min, max) per axis

Things to think about:
- When should the randomization be sampled?
- How large can the ranges be before the task becomes impossible?
- Which other quantities would be worth randomizing? (no need to implement them)

In [ ]:
def make_events(play: bool) -> dict[str, EventTermCfg]:
    return {
        # Example
        "reset_base": EventTermCfg(
            func=mdp.reset_root_pose_uniform,
            mode="reset",  # TODO: when should this event run?
            params={
                "pose_range": {
                    # TODO: (min, max) ranges for the initial pose
                    "x": (0.0, 0.0),  # m
                    "y": (0.0, 0.0),  # m
                    "z": (0.0, 0.0),  # m
                    "roll": (0.0, 0.0),  # rad
                    "pitch": (0.0, 0.0),  # rad
                    "yaw": (0.0, 0.0),  # rad
                }
            },
        ),
        # TODO: initialize joint angles to nominal pose
        # Domain randomization: randomize base CoM
        "base_com": EventTermCfg(
            mode="reset", # TODO: when should domain randomization be applied?
            func=mdp.body_com_offset,
            params={
                "asset_cfg": SceneEntityCfg("robot", body_names=("base_link",)),
                "operation": "add", # TODO: what type of operation
                "ranges": { # TODO: define your ranges
                    0: (0.0, 0.0), # m
                    1: (0.0, 0.0), # m
                    2: (0.0, 0.0), # m
                },
            },
        ),
    }

### 3.1g Terrain curriculum

The stairs terrain is a grid of staircases: 10 rows of increasing difficulty (step height from 0 cm to 20 cm) and 2 columns (pyramids to walk down, and inverted pyramids to climb up).
Each robot is assigned a row, its **terrain level**.

Without a curriculum, most robots would face stairs they cannot climb yet: they fall immediately, collect almost no reward, and learn nothing.
With `terrain_levels_vel`, the level adapts to each robot's performance at the end of every episode:

- **promoted** to a harder row if it walked at least 80 % of the distance its command asked for OR it reached the end of the terrain
- **demoted** to an easier row if it walked less than half of that distance.

Follow `Curriculum/terrain_levels/...` in TensorBoard: a rising mean level means the robots master higher steps.
Training has succeeded when most robots stay on the hardest levels.

Things to think about:
- Why does the curriculum use the walked distance rather than the reward?
- What happens if robots are commanded to stand still?
- What other types of curriculums can you think about? (no need to implement them)

In [ ]:
def make_curriculum(play: bool) -> dict[str, CurriculumTermCfg]:
    return {
        "terrain_levels": CurriculumTermCfg(
            func=mdp.terrain_levels_vel,
            params={"command_name": "joystick", "edge_term_name": "terrain_edge_reached"},
        ),
    }

### 3.1h Assembly

As in Task 1, with `terrain="stairs"` and the new `curriculum` part.

In [ ]:
def make_env_cfg(play: bool = False, seed: int = 42) -> ManagerBasedRlEnvCfg:
    return ManagerBasedRlEnvCfg(
        scene=scene.make_scene_cfg(
            num_envs=1 if play else 4096,
            terrain="stairs", # We are using the stairs environment this time!
            robot_entity_name="robot",
        ),
        sim=scene.make_sim_cfg(timestep_s=0.005),
        actions=make_actions(play),
        commands=make_commands(play),
        curriculum=make_curriculum(play),
        events=make_events(play),
        observations=make_observations(play),
        rewards=make_rewards(play),
        terminations=make_terminations(play),
        decimation=4,
        episode_length_s=20.0,
        seed=seed,
    )

### 3.1d Visualize your environment

Check that robots spawn correctly on the stairs (make sure to pause and reset), that the terrain scan rays look right, and that a fallen robot is reset.
Stop the viewer with the **interrupt** button (■) before running the next cell.

In [ ]:
env_cfg = make_env_cfg(play=True)
play.interactive(env_cfg, policy="random") # "random" or "zero"

## 3.2 Building the training configuration

Same networks and PPO settings as in Task 1.
While better performance can be obtained by tuning these, we recommend keeping the parameters we tuned for you.

In [ ]:
def make_actor_cfg() -> RslRlModelCfg:
    return RslRlModelCfg(
        hidden_dims=(128, 128, 128),
        activation="elu",
        obs_normalization=True,
        distribution_cfg={
            "class_name": "GaussianDistribution",
            "init_std": 1.0,
            "std_type": "scalar",
        },
    )

def make_critic_cfg() -> RslRlModelCfg:
    return RslRlModelCfg(
        hidden_dims=(128, 128, 128),
        activation="elu",
        obs_normalization=True,
    )

def make_ppo_cfg() -> RslRlPpoAlgorithmCfg:
    return RslRlPpoAlgorithmCfg(
        value_loss_coef=1.0,
        use_clipped_value_loss=True,
        clip_param=0.2,
        entropy_coef=0.01,
        num_learning_epochs=5,
        num_mini_batches=4,
        learning_rate=1.0e-3,
        schedule="adaptive",
        gamma=0.995,
        lam=0.97,
        desired_kl=0.01,
        max_grad_norm=1.0,
    )

def make_rl_cfg(seed: int = 42) -> RslRlOnPolicyRunnerCfg:
    return RslRlOnPolicyRunnerCfg(
        seed=seed,
        num_steps_per_env=24,
        max_iterations=500,
        save_interval=500,
        actor=make_actor_cfg(),
        critic=make_critic_cfg(),
        algorithm=make_ppo_cfg(),
    )

## 3.3 Training

This cell starts TensorBoard (open the printed link) and then trains.
Besides the reward and episode length curves, follow the terrain level (`Curriculum/terrain_levels/...`) to see how far up the curriculum the robots get.

In [ ]:
train.tensorboard()

train.train(
    env_cfg=make_env_cfg(),
    rl_cfg=make_rl_cfg(),
    experiment="stairs",
    variant="baseline",
)

## 3.4 Testing your policy

Watch your policy in the viewer and check that it crosses stairs **reliably**: up and down, over many episodes rather than once.
If it fails, note *how*: tripping on step edges, refusing to move, slipping, falling when stepping down.
Each failure points to a different fix in the observations, rewards or terminations.

In [ ]:
run = registry.runs(experiment="stairs", variant="baseline")[-1]
print(f"Found checkpoint at: {run.checkpoint}")

play.interactive(
    env_cfg=make_env_cfg(play=True),
    rl_cfg=make_rl_cfg(),
    policy=run.checkpoint,
)